In [1]:
!unzip -q data.zip

replace data/labels.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [17]:
!apt-get update
!apt-get install -y tesseract-ocr

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,899 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

In [19]:
!find /usr/share -name tessdata

/usr/share/tesseract-ocr/4.00/tessdata


In [20]:
!sudo mv urd.traineddata /usr/share/tesseract-ocr/4.00/tessdata/

In [21]:
!tesseract --list-langs

List of available languages (3):
eng
osd
urd


In [22]:
import cv2
import pytesseract

image_path = "/content/data/raw/others/0.png"

img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

text = pytesseract.image_to_string(
    gray,
    lang="urd",
    config="--psm 6"
)

print(text)

ModuleNotFoundError: No module named 'pytesseract'

In [23]:
!pip install -q pytesseract opencv-python pillow pandas jiwer

In [24]:
import pytesseract
import cv2
import pandas as pd
from PIL import Image

print("Pytesseract installed successfully!")

Pytesseract installed successfully!


In [25]:
import pytesseract

print(pytesseract.get_tesseract_version())

4.1.1


In [27]:
import cv2
import pytesseract

# Agar zarurat ho to tesseract ka path set karo
pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

image_path = "/content/data/raw/other/0.png"

img = cv2.imread(image_path)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

text = pytesseract.image_to_string(
    gray,
    lang="urd",
    config="--psm 6"
)

print(text)

اور بنوں ( ما تمرو جنگ , اے ایف کی ) بنوں میں اثوام مب راٗییل اور



In [28]:
import cv2

def preprocess(image_path):
    img = cv2.imread(image_path)

    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Denoise
    gray = cv2.fastNlMeansDenoising(gray)

    # Resize 2x (better OCR)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # Threshold
    gray = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    return gray

In [29]:
image = preprocess("/content/data/raw/other/0.png")

text = pytesseract.image_to_string(
    image,
    lang="urd",
    config="--oem 3 --psm 6"
)

print(text)

اور ہنوں ( نما تحدروججنگٹ, اے ایف پی) بننوں میں اثوام مب اتیل اور



In [32]:
import pandas as pd

labels = pd.read_csv("/content/data/labels.csv")

# Correct folder name
labels["image"] = labels["image"].str.replace(
    "data/raw/others/",
    "data/raw/other/",
    regex=False
)

print(labels.head())

                  image                                               text
0  data/raw/other/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1  data/raw/other/1.png        اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس
2  data/raw/other/2.png  کی لوڈ شیڈنگ کیخلاف مظاہرہ کیا۔ پولیس تشدد سے ...
3  data/raw/other/3.png  منشور علی جاں بحق اور 2 خواتین سمیت 14 افراد ز...
4  data/raw/other/4.png  علاقے میں کرفیو نافذ کر دیا گیا ہے۔ مکینوں نے ...


In [39]:
predictions = []

for path in labels["image"]:

    image_path = "/content/" + path

    img = preprocess(image_path)

    if img is None:
        predictions.append("")
        continue

    pred = pytesseract.image_to_string(
        img,
        lang="urd",
        config="--oem 3 --psm 6"
    )

    predictions.append(pred)

labels["Prediction"] = predictions

labels.to_csv("/content/tesseract_results.csv", index=False)

print(labels[["image","text","Prediction"]].head())

                  image                                               text  \
0  data/raw/other/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...   
1  data/raw/other/1.png        اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس   
2  data/raw/other/2.png  کی لوڈ شیڈنگ کیخلاف مظاہرہ کیا۔ پولیس تشدد سے ...   
3  data/raw/other/3.png  منشور علی جاں بحق اور 2 خواتین سمیت 14 افراد ز...   
4  data/raw/other/4.png  علاقے میں کرفیو نافذ کر دیا گیا ہے۔ مکینوں نے ...   

                                          Prediction  
0  اور ہنوں ( نما تحدروججنگٹ, اے ایف پی) بننوں می...  
1           کے سا تھ مکتہ عا تے کے عوام نے می وکس\n  
2  کی لوڈ شی ڈ۰ لاف مظاہر ہکیا۔ لیٹس تتمردے ایک و...  
3  نشور علی جاں بن اور 2 خو ا تن سیت 14 افراوز تھ...  
4  ات میں کر ٹیو ماف کرد ما گیا سے میینوں نے الٹر...  


In [40]:
from jiwer import cer, wer

correct = 0
total_cer = 0
total_wer = 0

for gt, pred in zip(labels["text"], labels["Prediction"]):

    gt = str(gt).strip()
    pred = str(pred).strip()

    if gt == pred:
        correct += 1

    total_cer += cer(gt, pred)
    total_wer += wer(gt, pred)

accuracy = (correct / len(labels)) * 100
avg_cer = total_cer / len(labels)
avg_wer = total_wer / len(labels)

print("=" * 50)
print("Total Images :", len(labels))
print("Correct      :", correct)
print(f"Accuracy     : {accuracy:.2f}%")
print(f"CER          : {avg_cer:.4f}")
print(f"WER          : {avg_wer:.4f}")

Total Images : 180
Correct      : 0
Accuracy     : 0.00%
CER          : 0.4437
WER          : 0.9217


In [41]:
from difflib import SequenceMatcher

scores = []

for gt, pred in zip(labels["text"], labels["Prediction"]):
    scores.append(SequenceMatcher(None, str(gt), str(pred)).ratio())

similarity = sum(scores) / len(scores) * 100

print(f"Similarity Accuracy: {similarity:.2f}%")

Similarity Accuracy: 59.71%


In [38]:
import os
import cv2

for i, path in enumerate(labels["image"]):

    image_path = "/content/" + path

    if not os.path.exists(image_path):
        print(f"[{i}] File does not exist: {image_path}")
        break

    img = cv2.imread(image_path)

    if img is None:
        print(f"[{i}] OpenCV cannot read: {image_path}")
        break

print("Done checking.")

Done checking.


In [35]:
import pandas as pd

labels = pd.read_csv("/content/data/labels.csv")

# Fix all incorrect folder names
labels["image"] = labels["image"].str.replace(
    "data/raw/others/",
    "data/raw/other/",
    regex=False
)

labels["image"] = labels["image"].str.replace(
    "data/raw/newspapers/",
    "data/raw/newspaper/",
    regex=False
)

# Save corrected labels (optional)
labels.to_csv("/content/data/labels_fixed.csv", index=False)

print(labels.head())

                  image                                               text
0  data/raw/other/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1  data/raw/other/1.png        اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس
2  data/raw/other/2.png  کی لوڈ شیڈنگ کیخلاف مظاہرہ کیا۔ پولیس تشدد سے ...
3  data/raw/other/3.png  منشور علی جاں بحق اور 2 خواتین سمیت 14 افراد ز...
4  data/raw/other/4.png  علاقے میں کرفیو نافذ کر دیا گیا ہے۔ مکینوں نے ...


In [37]:
import pandas as pd

labels = pd.read_csv("/content/data/labels.csv")

replacements = {
    "data/raw/others/": "data/raw/other/",
    "data/raw/newspapers/": "data/raw/newspaper/",
    "data/raw/signboard/": "data/raw/signboards/",
    "data/raw/book/": "data/raw/books/",
}

for old, new in replacements.items():
    labels["image"] = labels["image"].str.replace(old, new, regex=False)

labels.to_csv("/content/data/labels_fixed.csv", index=False)

print("Folder names corrected.")

Folder names corrected.


In [42]:
import gradio as gr
import cv2
import pytesseract

pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

def preprocess(image):

    img = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray = cv2.fastNlMeansDenoising(gray)

    gray = cv2.resize(
        gray,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    return gray


def extract_text(image):

    if image is None:
        return "Please upload an Urdu image."

    img = preprocess(image)

    text = pytesseract.image_to_string(
        img,
        lang="urd",
        config="--oem 3 --psm 6"
    )

    if text.strip() == "":
        return "No Urdu text detected."

    return text


demo = gr.Interface(
    fn=extract_text,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Textbox(lines=10),
    title="Urdu OCR",
    description="Upload an Urdu image and extract text using Tesseract OCR."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9bf07e2920614b62f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
